# Module 9: Word2Vec — Austen vs. Melville

DS 5001 | Clay Harris

We create separate word embeddings for Austen and Melville using word2vec, visualize with t-SNE, and explore semantic algebra.

---
# Set Up

## Configs

In [1]:
OHCO = ['book_id', 'chap_num', 'para_num', 'sent_num', 'token_num']
BAG = OHCO[:3]  # Paragraphs
window = 5

## Imports

In [2]:
import pandas as pd
import numpy as np
from gensim.models import word2vec
from sklearn.manifold import TSNE
import plotly.express as px

---
# Load Data

In [3]:
LIB = pd.read_csv('LIB.csv')
TOKEN = pd.read_csv('TOKEN2.csv').set_index(OHCO)
TOKEN.head()

pos_tuple  pos  \
book_id chap_num para_num sent_num token_num                              
158     1        1        0        0               ('Emma', 'NNP')  NNP   
                                   1          ('Woodhouse', 'NNP')  NNP   
                                   3            ('handsome', 'NN')   NN   
                                   5              ('clever', 'NN')   NN   
                                   7                 ('and', 'CC')   CC   

                                              token_str   term_str  term_id  
book_id chap_num para_num sent_num token_num                                 
158     1        1        0        0               Emma       emma    11614  
                                   1          Woodhouse  woodhouse    39340  
                                   3           handsome   handsome    15924  
                                   5             clever     clever     6354  
                                   7                and        and     1426

In [4]:
austen_ids = LIB[LIB.author == 'austen'].book_id.tolist()
melville_ids = LIB[LIB.author == 'melville'].book_id.tolist()
print('Austen book_ids:', austen_ids)
print('Melville book_ids:', melville_ids)

Austen book_ids: [158, 946, 1212, 141, 121, 105, 1342, 161]
Melville book_ids: [15422, 13720, 13721, 2701, 4045, 34970, 8118, 53861, 21816, 15859, 1900, 10712]


---
# Build Gensim Corpora

We split the TOKEN table by author and convert each into a list of word lists (one per paragraph), excluding proper nouns.

In [5]:
def make_corpus(tokens):
    return tokens[~tokens.pos.str.match('NNPS?')]\
        .groupby(BAG)\
        .term_str.apply(lambda x: x.tolist())\
        .reset_index()['term_str'].tolist()

In [6]:
austen_corpus = make_corpus(TOKEN.loc[TOKEN.index.get_level_values('book_id').isin(austen_ids)])
melville_corpus = make_corpus(TOKEN.loc[TOKEN.index.get_level_values('book_id').isin(melville_ids)])
print(f'Austen paragraphs: {len(austen_corpus)}')
print(f'Melville paragraphs: {len(melville_corpus)}')

Austen paragraphs: 10449
Melville paragraphs: 20086


---
# Create Word2Vec Models

In [7]:
austen_model = word2vec.Word2Vec(austen_corpus, vector_size=246, window=window, min_count=100, workers=4)
melville_model = word2vec.Word2Vec(melville_corpus, vector_size=246, window=window, min_count=100, workers=4)
print(f'Austen vocab size: {len(austen_model.wv)}')
print(f'Melville vocab size: {len(melville_model.wv)}')

Austen vocab size: 750
Melville vocab size: 1130


---
# t-SNE Visualizations

We filter the vocabulary to content words (nouns and adjectives by majority POS tag) for visualization. The full model is retained for analogies.

## Build Content Word Sets

In [8]:
def get_content_words(tokens):
    """Words whose most common POS tag is a noun or adjective."""
    pos_counts = tokens.groupby(['term_str', 'pos']).size().reset_index(name='count')
    majority_pos = pos_counts.loc[pos_counts.groupby('term_str')['count'].idxmax()]
    return set(majority_pos[majority_pos.pos.isin({'NN', 'NNS', 'JJ', 'JJR', 'JJS'})].term_str)

austen_tokens = TOKEN.loc[TOKEN.index.get_level_values('book_id').isin(austen_ids)]
melville_tokens = TOKEN.loc[TOKEN.index.get_level_values('book_id').isin(melville_ids)]

austen_content = get_content_words(austen_tokens)
melville_content = get_content_words(melville_tokens)

In [9]:
def make_tsne_coords(model, content_words, perplexity=30, n_iter=2500):
    """Build t-SNE coordinates for content words only."""
    words = [w for w in model.wv.key_to_index if w in content_words]
    vectors = np.array([model.wv.get_vector(w) for w in words])
    coords = pd.DataFrame({'label': words})
    tsne = TSNE(perplexity=perplexity, n_components=2, init='pca', n_iter=n_iter, random_state=23)
    tsne_vals = tsne.fit_transform(vectors)
    coords['x'] = tsne_vals[:, 0]
    coords['y'] = tsne_vals[:, 1]
    return coords

## Austen t-SNE

In [10]:
austen_coords = make_tsne_coords(austen_model, austen_content)
print(f'Content words plotted: {len(austen_coords)}')
austen_coords.head()

/opt/anaconda3/lib/python3.12/site-packages/sklearn/manifold/_t_sne.py:1162: FutureWarning: 'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.
  warnings.warn(


Content words plotted: 369


,label,x,y
0,such,3.408139,12.779723
1,much,4.734151,8.100683
2,own,-12.387075,-6.883584
3,good,7.774528,13.759757
4,time,8.167141,-12.894143


In [11]:
px.scatter(austen_coords, 'x', 'y', text='label', height=1000, width=1200,
           title='Austen Word Embeddings — Content Words (t-SNE)').update_traces(mode='text')

## Melville t-SNE

In [12]:
melville_coords = make_tsne_coords(melville_model, melville_content)
print(f'Content words plotted: {len(melville_coords)}')
melville_coords.head()

/opt/anaconda3/lib/python3.12/site-packages/sklearn/manifold/_t_sne.py:1162: FutureWarning:

'n_iter' was renamed to 'max_iter' in version 1.5 and will be removed in 1.7.



Content words plotted: 637


,label,x,y
0,man,26.384312,0.408061
1,old,-0.013656,-27.169380
2,other,-2.255477,-19.670256
3,time,-7.460733,-9.195116
4,little,0.329496,2.645949


In [13]:
px.scatter(melville_coords, 'x', 'y', text='label', height=1000, width=1200,
           title='Melville Word Embeddings — Content Words (t-SNE)').update_traces(mode='text')

---
# Cluster Analysis

## Austen

The Austen t-SNE is largely a diffuse cloud — most content words don't separate into sharp regions. Two clusters do stand out:

1. **Family & Kinship**: A tight cluster of *mother, brother, cousin, aunt, sister, father, uncle, daughter, son*. These kinship terms group closely because they co-occur in Austen's narration of family dynamics and domestic negotiations. The tightness of this cluster relative to the rest of the plot reflects how central and internally consistent this vocabulary is in her prose.

2. **Meals & Domestic Time**: A cluster of *dinner, breakfast, table, evening, room*. These words mark the daily rhythms of Austen's social world — meals structure the plot, and rooms are where conversation (and therefore plot) happens. Their clustering reflects how tightly bound eating, time of day, and domestic space are in her novels.

The lack of other sharp clusters is itself informative: Austen's vocabulary is broadly social and evaluative, and many of her content words (*agreeable*, *kind*, *pleasure*, *manner*) are used across contexts rather than confined to specific semantic domains.

## Melville

The Melville t-SNE shows clearer large-scale structure than Austen's, with several identifiable regions:

1. **Environmental & Maritime**: A large cluster of *deck, sea, land, whale, frigate, ship, water, sail, boat, mast, wind*. This is the dominant semantic region in Melville's embedding space — the physical world of ocean, ships, and creatures. Its size relative to other clusters reflects how much of his prose is devoted to describing the maritime environment.

2. **Abstract & Philosophical**: A broad cloud of words like *dream, wisdom, free, play, peace, confidence, answer, hope, mind*. These are the more interior, philosophical terms that accompany Melville's metaphysical digressions. The cluster is looser than the maritime one because these abstract concepts are used in varied narrative contexts.

3. **Time & Counting**: A sharp, tight cluster of *hours, days, years, twenty, dollars, minutes*. Numerical and temporal terms group closely because they appear in similar syntactic contexts (quantities, durations, measurements). This is the most visually distinct cluster in the Melville plot.

4. **People & Social Roles**: A somewhat loose but identifiable cluster of *girls, boys, fellows, young, old, officers, sailors, men, natives, kings, gods*. This groups the humans (and deities) of Melville's world — from the ship's crew to the island populations to the cosmic figures of his allegories.

---
# Semantic Algebra

$A : B :: C : D? \rightarrow B - A + C = D$

In [14]:
def complete_analogy(A, B, C, model, n=3):
    try:
        return model.wv.most_similar(positive=[B, C], negative=[A])[0:n]
    except KeyError as e:
        print('Error:', e)
        return None

## Austen Analogies

In [15]:
# man : gentleman :: woman : ?
print('man : gentleman :: woman : ?')
complete_analogy('man', 'gentleman', 'woman', austen_model)

man : gentleman :: woman : ?


[('amiable', 0.7741524577140808),
 ('girl', 0.7666923403739929),
 ('lady', 0.7289682030677795)]

In [16]:
# husband : wife :: brother : ?
print('husband : wife :: brother : ?')
complete_analogy('husband', 'wife', 'brother', austen_model)

husband : wife :: brother : ?


[('cousin', 0.7579434514045715),
 ('son', 0.7425504922866821),
 ('father', 0.6653763055801392)]

In [17]:
# daughter : mother :: son : ?
print('daughter : mother :: son : ?')
complete_analogy('daughter', 'mother', 'son', austen_model)

daughter : mother :: son : ?


[('uncle', 0.848430871963501),
 ('brother', 0.8400045037269592),
 ('cousin', 0.837056040763855)]

## Melville Analogies

In [18]:
# sea : ship :: land : ?
print('sea : ship :: land : ?')
complete_analogy('sea', 'ship', 'land', melville_model)

sea : ship :: land : ?


[('frigate', 0.5920838117599487),
 ('voyage', 0.5482825040817261),
 ('vessel', 0.5472679138183594)]

In [19]:
# man : captain :: woman : ?
print('man : captain :: woman : ?')
complete_analogy('man', 'captain', 'woman', melville_model)

man : captain :: woman : ?


[('girl', 0.6171871423721313),
 ('wife', 0.5539141297340393),
 ('son', 0.5519962906837463)]

In [20]:
# ship : captain :: island : ?
print('ship : captain :: island : ?')
complete_analogy('ship', 'captain', 'island', melville_model)

ship : captain :: island : ?


[('presence', 0.6415110230445862),
 ('office', 0.6120032072067261),
 ('unfortunate', 0.6062923073768616)]

---
# Interpretation

## Austen Analogies

**man : gentleman :: woman : amiable/girl/lady**. *Lady* appears as the third-ranked result (0.729), which is the correct gendered social rank mapping. The top result *amiable* (0.774) is telling — in Austen, the female equivalent of "gentleman" bleeds into character evaluation. A gentleman is defined by conduct; the model associates the female analogue with being *amiable*, reflecting how women in these novels are judged by temperament rather than title alone.

**husband : wife :: brother : cousin/son/father**. The model stays entirely within the family domain. The top result *cousin* (0.758) makes sense — if husband and wife are a paired domestic relationship, the model maps *brother* to *cousin*, another same-generation familial pairing. *Son* and *father* round out the family structure.

**daughter : mother :: son : uncle/brother/cousin** (top score 0.848). The model correctly maps into the male family role space. All three results are male kinship terms, confirming the model's grasp of gendered family structure learned purely from co-occurrence patterns.

## Melville Analogies

**sea : ship :: land : frigate/voyage/vessel**. The model cannot escape the nautical domain — even when prompted with "land," it returns maritime vocabulary. This reflects how thoroughly Melville's prose is saturated with seafaring language. The concept of "land" in his novels is defined in relation to the sea.

**man : captain :: woman : girl/wife/son**. The model maps the authority relationship to gendered terms. *Girl* (0.617) and *wife* (0.554) suggest that women in Melville's novels are defined primarily by youth or marital status rather than by occupational authority — a reflection of the genre, where female characters rarely hold command roles.

**ship : captain :: island : presence/office/unfortunate**. A less clean result than the others. *Office* (0.612) is the most interesting return — it captures the abstract concept of authority or position, suggesting the model grasps that the captain-ship relationship is one of command even if it cannot name the specific island authority figure (chief/king). *Presence* may reflect how island leaders in Melville are often described by their imposing bearing.

## Overall

The two models reflect fundamentally different semantic worlds. Austen's embeddings are organized around social relationships, emotional states, and interpersonal evaluation. Melville's are organized around physical space, nature, and hierarchies of power. This is consistent with our earlier topic models, where Austen's dominant topics were family and social conduct, while Melville's were nautical life and the natural world.